In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
import sys
import pandas as pd
import altair as alt
sys.path.append("..")
from budget.grist import get_records, records_to_df


In [ ]:
# lecture des données sur Grist
df_conso = records_to_df(get_records("Consommations"))
df_periode = records_to_df(get_records("Periode"))
df_conso_periode = df_conso.merge(df_periode, left_on="periode", right_on="id", suffixes=("", "_periode"))

In [60]:
# mise en forme (annee, statut des financements)
df_conso_periode['annee'] = pd.to_datetime(df_conso_periode['mois'], unit='s', utc=True).dt.year
df_conso_periode['statut'] = 'consommée'
# ajout de la ligne prévisionnelle
conso_previsionnelle = [{'annee': 2026, 'montant_ttc': 200000, 'statut': 'prévisionnel'},
                        {'annee': 2027, 'montant_ttc': 400000, 'statut': 'prévisionnel'}]
df_conso_periode = pd.concat([df_conso_periode, pd.DataFrame(conso_previsionnelle)], ignore_index=True)
df_conso_annuelle = (
    df_conso_periode
    .set_index('annee')['montant_ttc']
    .groupby('annee')
    .sum()
    .reset_index()
)

In [70]:
chart_conso = alt.Chart(df_conso_annuelle).mark_bar(size=40).encode(
    x=alt.X('annee:O', title='Année'),
    y=alt.Y('montant_ttc:Q', title='Montant TTC (€)', scale=alt.Scale(domain=[0, 400000])),
    tooltip=['annee:O', 'montant_ttc:Q'],
).properties(
    title='Consommations annuelles (historique + prévisionnel)',
    width=300)
chart_conso

alt.Chart(...)

In [75]:
# dataframe du sponsoring
df_sponsor = pd.DataFrame([{'annee': 2024, 'montant_ttc': 15552, 'sponsor': 'Cerema'},
                          {'annee': 2025, 'montant_ttc': 167184, 'sponsor': 'Cerema'},
                          {'annee': 2025, 'montant_ttc': 44539, 'sponsor': 'ANCT'},
                          {'annee': 2026, 'montant_ttc': 163814, 'sponsor': 'ANCT'},
                          {'annee': 2026, 'montant_ttc': 105000, 'sponsor': 'Cerema'},
                          {'annee': 2027, 'montant_ttc': 400000, 'sponsor': 'à définir'}])

df_recette = pd.DataFrame([{'annee': 2024, 'montant_ttc': 0, 'client': 'aucun'},
                          {'annee': 2025, 'montant_ttc': 0, 'client': 'aucun'},
                          {'annee': 2026, 'montant_ttc': 25000, 'client': 'Collectivités territoriales'},
                          {'annee': 2026, 'montant_ttc': 40000, 'client': 'DGPR'},
                          {'annee': 2027, 'montant_ttc': 150000, 'client': 'DGPR'},
                          {'annee': 2027, 'montant_ttc': 100000, 'client': 'Collectivités territoriales'}
                          ])

In [79]:
chart_sponsor = alt.Chart(df_sponsor).mark_bar(size=40).encode(
    x=alt.X('annee:O', title='Année'),
    y=alt.Y('montant_ttc:Q', title='Montant TTC (€)', scale=alt.Scale(domain=[0, 400000])),
    color=alt.Color('sponsor:N', title='Sponsor', legend=alt.Legend(orient='top-left')),
    tooltip=['annee:O', 'sponsor:N', 'montant_ttc:Q'],
).properties(
    title='Sponsoring annuel',
    width=300)
chart_sponsor


alt.Chart(...)

In [80]:
chart_recette = alt.Chart(df_recette).mark_bar(size=40).encode(
    x=alt.X('annee:O', title='Année'),
    y=alt.Y('montant_ttc:Q', title='Montant TTC (€)', scale=alt.Scale(domain=[0, 400000])),
    color=alt.Color('client:N', title='Client', legend=alt.Legend(orient='top-left')),
    tooltip=['annee:O', 'client:N', 'montant_ttc:Q'],
).properties(
    title='Recettes annuelles par client',
    width=300)
chart_recette


alt.Chart(...)

In [66]:
df_conso_annuelle

,annee,montant_ttc
0,2024.0,15552.0
1,2025.0,189489.6
2,2026.0,386048.0
3,2027.0,400000.0
